# Task 1 — Ingest with COPY INTO (Bronze layer)

**The simple idea:**

* The volume `demo1.bronze.source_files` is a **drop zone** — throw any CSV (with the standard header) into it.
* `COPY INTO` loads **every NEW file in the volume exactly once** — Databricks remembers which files it already loaded, so re-running the job never re-ingests old files.
* After loading, every new row is stamped with an **`ingested_at` timestamp**.
* "Is there new data?" becomes one simple question: **how many rows were ingested in the last N minutes?** — Task 2 answers it using the `window_minutes` parameter.

In [0]:
# CELL 1 — Parameters
# "catalog" is passed by the job (point the whole demo at another
# catalog without touching any code).
# "job_id" / "run_id" are the run's REAL identifiers — the job passes
# them as dynamic values ({{job_id}} / {{run_id}}). They go into the
# audit log below, so every log row can be traced to a job run.
dbutils.widgets.text("catalog", "demo1", "Catalog that holds all demo tables")
dbutils.widgets.text("job_id", "interactive", "Job ID (passed by the job)")
dbutils.widgets.text("run_id", "interactive", "Run ID (passed by the job)")

catalog = dbutils.widgets.get("catalog")
job_id = dbutils.widgets.get("job_id")
run_id = dbutils.widgets.get("run_id")

BRONZE_TABLE = f"{catalog}.bronze.sales_orders"
VOLUME_PATH = f"/Volumes/{catalog}/bronze/source_files/"  # the drop zone (ALL csv files)

print(f"catalog      : {catalog}")
print(f"job_id       : {job_id}")
print(f"run_id       : {run_id}")
print(f"bronze table : {BRONZE_TABLE}")
print(f"source       : {VOLUME_PATH} (all NEW csv files here)")

catalog      : demo1
job_id       : interactive
run_id       : interactive
bronze table : demo1.bronze.sales_orders
source       : /Volumes/demo1/bronze/source_files/ (all NEW csv files here)


In [0]:
# CELL 2 — COPY INTO: load every NEW file from the volume into bronze
# WHAT IS HAPPENING:
#   * CREATE TABLE (only once) defines the bronze table: the 7 CSV columns
#     as strings (raw layer = no casting yet) + one ingested_at column.
#   * COPY INTO loads only the files it has NOT loaded into this table
#     before (Databricks tracks them). Already-loaded files are skipped
#     automatically -> "only new data", zero bookkeeping.
#   * New rows arrive with ingested_at = NULL — the next cell stamps them.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
        order_id STRING, customer_name STRING, product STRING, category STRING,
        quantity STRING, unit_price STRING, order_date STRING,
        ingested_at TIMESTAMP
    )
""")

result = spark.sql(f"""
    COPY INTO {BRONZE_TABLE}
    FROM '{VOLUME_PATH}'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true')
""").collect()[0]

rows_loaded = result["num_inserted_rows"]
print(f"COPY INTO loaded {rows_loaded} NEW rows from the volume into {BRONZE_TABLE}")

In [0]:
# CELL 3 — Stamp the new rows with the ingestion timestamp
# WHAT IS HAPPENING:
#   * Only rows COPY INTO just loaded have ingested_at = NULL.
#   * One UPDATE stamps them all with now().
#   * From this moment: "new data" = rows with ingested_at in the last
#     N minutes. That single rule drives validate, transform and notify.
from pyspark.sql import functions as F

spark.sql(f"UPDATE {BRONZE_TABLE} SET ingested_at = current_timestamp() WHERE ingested_at IS NULL")

print("Latest rows in bronze (with their ingestion timestamp):")
display(spark.table(BRONZE_TABLE).orderBy(F.col("ingested_at").desc()).limit(15))

In [0]:
# CELL 4 — Audit log: the demo's diary (one row per task per run,
# including the real job_id / run_id — your run-level log book)
from datetime import datetime

LOG_TABLE = f"{catalog}.bronze.job_run_log"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
        logged_at TIMESTAMP, job_id STRING, run_id STRING, task_name STRING,
        rows_processed BIGINT, status STRING, message STRING
    )
""")

log_df = spark.createDataFrame(
    [(datetime.now(), job_id, run_id, "ingest", int(rows_loaded), "SUCCESS",
      f"COPY INTO loaded {rows_loaded} new rows from the volume")],
    "logged_at TIMESTAMP, job_id STRING, run_id STRING, task_name STRING, "
    "rows_processed BIGINT, status STRING, message STRING",
)
log_df.write.mode("append").saveAsTable(LOG_TABLE)
print("Audit row written to", LOG_TABLE)

## Task 1 — recap

* `COPY INTO` ingested every **new** CSV from the volume into `<catalog>.bronze.sales_orders` — already-loaded files were skipped automatically (verified: a second COPY INTO loads 0 rows).
* Every new row now carries `ingested_at` — the timestamp that answers "what is new data?".
* The job runs daily: each run ingests whatever new files arrived since the last run; Task 2 counts rows from the **last `window_minutes` minutes**.